# Aula 2 – Vídeo 4: Planejamento de Viagem (com condicionais)
Este notebook demonstra como usar LangGraph para estruturar um fluxo de planejamento de viagem,
com ramificações condicionais para transporte e hospedagem.

## 1) Imports e Setup

In [1]:
import os
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END

try:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import PromptTemplate
except Exception:
    ChatOpenAI, PromptTemplate = None, None

load_dotenv()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0) if ChatOpenAI else None

## 2) Definição do Estado Compartilhado

In [2]:
class State(TypedDict, total=False):
    destino: str
    origem: str
    orcamento: str
    perfil: str
    tipo_viagem: str
    distancia_km: int
    transporte: str
    hospedagem: str
    plano_final: str

## 3) Definição dos Nós

In [3]:
def normalizar_entrada(s: State) -> State:
    d = (s.get("destino") or "").strip() or "Lisboa"
    orc = s.get("orcamento", "medio").lower()
    perfil = s.get("perfil", "economia").lower()
    return {"destino": d, "orcamento": orc, "perfil": perfil}

def enriquecer_destino(s: State) -> State:
    destino = s["destino"].lower()
    if destino in {"rio de janeiro", "são paulo", "salvador", "recife"}:
        tipo, dist = "nacional", 450 if destino in {"rio de janeiro", "são paulo"} else 1500
    else:
        tipo, dist = "internacional", 3000
    return {"tipo_viagem": tipo, "distancia_km": dist}

def rota_transporte(s: State) -> str:
    dist, orc, perfil = s.get("distancia_km", 3000), s.get("orcamento","medio"), s.get("perfil","economia")
    if dist < 500: return "curto_alcance"
    if orc == "baixo" or perfil in {"economia","mochilao"}: return "longo_economico"
    return "longo_conforto"

def sugerir_transporte_curto(s: State) -> State:
    return {"transporte": f"Ônibus ou carro até {s['destino']}."}

def sugerir_transporte_longo_economico(s: State) -> State:
    return {"transporte": f"Voo econômico para {s['destino']}."}

def sugerir_transporte_longo_conforto(s: State) -> State:
    return {"transporte": f"Voo premium para {s['destino']}."}

def sugerir_hospedagem_economica(s: State) -> State:
    return {"hospedagem": f"Hostel em {s['destino']}."}

def sugerir_hospedagem_intermediaria(s: State) -> State:
    return {"hospedagem": f"Hotel 3–4★ em {s['destino']}."}

def sugerir_hospedagem_luxo(s: State) -> State:
    return {"hospedagem": f"Hotel 5★ em {s['destino']}."}

def montar_plano(s: State) -> State:
    plano = (f"Destino: {s['destino']}\n"
             f"Tipo: {s.get('tipo_viagem')} (≈{s.get('distancia_km')} km)\n"
             f"Transporte: {s['transporte']}\n"
             f"Hospedagem: {s['hospedagem']}")
    return {"plano_final": plano}

## 4) Construção do Grafo

In [4]:
g = StateGraph(State)
g.add_node("normalizar_entrada", normalizar_entrada)
g.add_node("enriquecer_destino", enriquecer_destino)
g.add_node("sugerir_transporte_curto", sugerir_transporte_curto)
g.add_node("sugerir_transporte_longo_economico", sugerir_transporte_longo_economico)
g.add_node("sugerir_transporte_longo_conforto", sugerir_transporte_longo_conforto)
g.add_node("sugerir_hospedagem_economica", sugerir_hospedagem_economica)
g.add_node("sugerir_hospedagem_intermediaria", sugerir_hospedagem_intermediaria)
g.add_node("sugerir_hospedagem_luxo", sugerir_hospedagem_luxo)
g.add_node("montar_plano", montar_plano)

g.set_entry_point("normalizar_entrada")
g.add_edge("normalizar_entrada", "enriquecer_destino")
g.add_conditional_edges("enriquecer_destino", rota_transporte, {
    "curto_alcance": "sugerir_transporte_curto",
    "longo_economico": "sugerir_transporte_longo_economico",
    "longo_conforto": "sugerir_transporte_longo_conforto",
})
g.add_edge("sugerir_transporte_curto", "sugerir_hospedagem_intermediaria")
g.add_edge("sugerir_transporte_longo_economico", "sugerir_hospedagem_economica")
g.add_edge("sugerir_transporte_longo_conforto", "sugerir_hospedagem_luxo")
g.add_edge("sugerir_hospedagem_economica", "montar_plano")
g.add_edge("sugerir_hospedagem_intermediaria", "montar_plano")
g.add_edge("sugerir_hospedagem_luxo", "montar_plano")
g.add_edge("montar_plano", END)

app = g.compile()
print(app.get_graph().draw_ascii())

                                                         +-----------+                                                          
                                                         | __start__ |                                                          
                                                         +-----------+                                                          
                                                               *                                                                
                                                               *                                                                
                                                               *                                                                
                                                    +--------------------+                                                      
                                                    | normalizar_entrada |                       

## 5) Execuções de Exemplo

In [5]:
print("=== Execução 1 (curto alcance, baixo orçamento) ===")
s1 = {"destino": "Rio de Janeiro", "orcamento": "baixo", "perfil": "mochilao"}
print(app.invoke(s1)["plano_final"])

print("\n=== Execução 2 (internacional, alto orçamento) ===")
s2 = {"destino": "Roma", "orcamento": "alto", "perfil": "conforto"}
print(app.invoke(s2)["plano_final"])

=== Execução 1 (curto alcance, baixo orçamento) ===
Destino: Rio de Janeiro
Tipo: nacional (≈450 km)
Transporte: Ônibus ou carro até Rio de Janeiro.
Hospedagem: Hotel 3–4★ em Rio de Janeiro.

=== Execução 2 (internacional, alto orçamento) ===
Destino: Roma
Tipo: internacional (≈3000 km)
Transporte: Voo premium para Roma.
Hospedagem: Hotel 5★ em Roma.
